In [94]:
import requests
import pandas as pd
import json

In [95]:
# the query protein
query = "CADM4_HUMAN"

In [ ]:
# Function that adds '_HUMAN' afterfix to column values
def add_suffix_human(value):
    return value + '_HUMAN'

In [105]:
# Function that takes a ppi dataframe and outputs from two columns only one column with the interactor

def get_interactors_for_target(df, col_a, col_b, target_protein):
    def get_interactors(row):
        if row[col_a] == target_protein:
            return row[col_b]
        elif row[col_b] == target_protein:
            return row[col_a]
        else:
            return None
    df['interactor_of_' + target_protein] = df.apply(get_interactors, axis = 1)
    
    return df

In [10]:
# Function that uses the UniProt-API to get the ProteinID or either the Protein name

def convert_protein_ID_Name(uniprotID_or_name): # e.g. can be P19320 or VCAM1_HUMAN
    
    uniprot_api_url = "https://rest.uniprot.org/uniprotkb" 
    format = "json"
    uniprot_request_url = f"{uniprot_api_url}/{uniprotID_or_name}?format={format}"
    uniprot_json = requests.get(uniprot_request_url).text
    
    uniprot_dict = json.loads(uniprot_json) # convering to dictionary

    if "_" in uniprotID_or_name:
        value = uniprot_dict["primaryAccession"]
    else:
        value = uniprot_dict['uniProtkbId']
        
    return (value)

# example
id_or_name = convert_protein_ID_Name(query)
print(id_or_name)

Q8NFZ8


In [96]:
# Extracting the data from the BioGRID API

# BioGRID Access Key: caf14dfd9a0b7be447d282c322b8362e

biogrid_api_url = "https://webservice.thebiogrid.org/interactions"

geneList = [query]

params = {
    "accesskey": "caf14dfd9a0b7be447d282c322b8362e", # need to request
    "format": "json",
    "geneList": geneList,
    "taxId": 9606, # human
    "max": 10000
}

response = requests.get(biogrid_api_url, params=params)
biogrid_interactions = response.json()

print(biogrid_interactions)

biogrid_data = {}
for interaction_id, interaction in biogrid_interactions.items():
    biogrid_data[interaction_id] = interaction
    biogrid_data[interaction_id]["INTERACTION_ID"] = interaction_id
    
# loading into dataframe

biogrid_df = pd.DataFrame.from_dict(biogrid_data, orient="index")

columns = [
    "INTERACTION_ID",
    "ENTREZ_GENE_A",
    "ENTREZ_GENE_B",
    "OFFICIAL_SYMBOL_A",
    "OFFICIAL_SYMBOL_B",
    "EXPERIMENTAL_SYSTEM",
    "PUBMED_ID",
    "PUBMED_AUTHOR",
    "THROUGHPUT",
    "QUALIFICATIONS"]

biogrid_df = biogrid_df[columns]

biogrid_df.head(5)

{'600990': {'BIOGRID_INTERACTION_ID': 600990, 'ENTREZ_GENE_A': '1454', 'ENTREZ_GENE_B': '199731', 'BIOGRID_ID_A': 107838, 'BIOGRID_ID_B': 128268, 'SYSTEMATIC_NAME_A': 'RP1-5O6.1', 'SYSTEMATIC_NAME_B': '-', 'OFFICIAL_SYMBOL_A': 'CSNK1E', 'OFFICIAL_SYMBOL_B': 'CADM4', 'SYNONYMS_A': 'CKIepsilon|HCKIE', 'SYNONYMS_B': 'IGSF4C|NECL4|Necl-4|TSLL2|synCAM4', 'EXPERIMENTAL_SYSTEM': 'Two-hybrid', 'EXPERIMENTAL_SYSTEM_TYPE': 'physical', 'PUBMED_AUTHOR': 'Vinayagam A (2011)', 'PUBMED_ID': 21900206, 'ORGANISM_A': 9606, 'ORGANISM_B': 9606, 'THROUGHPUT': 'High Throughput', 'QUANTITATION': '-', 'MODIFICATION': '-', 'ONTOLOGY_TERMS': {}, 'QUALIFICATIONS': '-', 'TAGS': '-', 'SOURCEDB': 'BIOGRID'}, '601286': {'BIOGRID_INTERACTION_ID': 601286, 'ENTREZ_GENE_A': '1780', 'ENTREZ_GENE_B': '199731', 'BIOGRID_ID_A': 108118, 'BIOGRID_ID_B': 128268, 'SYSTEMATIC_NAME_A': '-', 'SYSTEMATIC_NAME_B': '-', 'OFFICIAL_SYMBOL_A': 'DYNC1I1', 'OFFICIAL_SYMBOL_B': 'CADM4', 'SYNONYMS_A': 'DNCI1|DNCIC1', 'SYNONYMS_B': 'IGSF4C|N

,INTERACTION_ID,ENTREZ_GENE_A,ENTREZ_GENE_B,OFFICIAL_SYMBOL_A,OFFICIAL_SYMBOL_B,EXPERIMENTAL_SYSTEM,PUBMED_ID,PUBMED_AUTHOR,THROUGHPUT,QUALIFICATIONS
600990,600990,1454,199731,CSNK1E,CADM4,Two-hybrid,21900206,Vinayagam A (2011),High Throughput,-
601286,601286,1780,199731,DYNC1I1,CADM4,Two-hybrid,21900206,Vinayagam A (2011),High Throughput,-
1051917,1051917,6472,199731,SHMT2,CADM4,Affinity Capture-RNA,22658674,Castello A (2012),High Throughput,-
1196008,1196008,253012,199731,HEPACAM2,CADM4,Affinity Capture-MS,26186194,Huttlin EL (2015),High Throughput,BioPlex 1.0 HEK 293T cells CompPASS score = 0....
2232293,2232293,29785,199731,CYP2S1,CADM4,Affinity Capture-MS,28514442,Huttlin EL (2017),High Throughput,BioPlex 2.0 HEK 293T cells CompPASS score = 0....


In [108]:
# Cleaning the dataframe

# filter out the columns that are needed
biogrid_df_filter = biogrid_df[[    'OFFICIAL_SYMBOL_A', 
                                    'OFFICIAL_SYMBOL_B', 
                                    'EXPERIMENTAL_SYSTEM', 
                                    'PUBMED_ID',
                                    'PUBMED_AUTHOR']]

# adding 'HUMAN' afterfix to the column values
biogrid_df_filter[['OFFICIAL_SYMBOL_A', 'OFFICIAL_SYMBOL_B']] = biogrid_df_filter[['OFFICIAL_SYMBOL_A', 'OFFICIAL_SYMBOL_B']].apply(add_suffix_human)

# renaming column headers
df_biogrid_final = biogrid_df_filter.rename(columns= {  'OFFICIAL_SYMBOL_A': 'biogrid_interactor_a', 
                                                        'OFFICIAL_SYMBOL_B': 'biogrid_interactor_b',
                                                        'EXPERIMENTAL_SYSTEM': 'biogrid_method',
                                                        'PUBMED_ID': 'biogrid_pubID',
                                                        'PUBMED_AUTHOR': 'biogrid_publication'})

# get one column, only the interactor of the query
df_biogrid = get_interactors_for_target(df_biogrid_final, 'biogrid_interactor_a', 'biogrid_interactor_b', query)

df_biogrid.head(5)


# df_biogrid_final.head(5)

/tmp/ipykernel_231037/2593118694.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  biogrid_df_filter[['OFFICIAL_SYMBOL_A', 'OFFICIAL_SYMBOL_B']] = biogrid_df_filter[['OFFICIAL_SYMBOL_A', 'OFFICIAL_SYMBOL_B']].apply(add_suffix_human)


,biogrid_interactor_a,biogrid_interactor_b,biogrid_method,biogrid_pubID,biogrid_publication,interactor_of_CADM4_HUMAN
600990,CSNK1E_HUMAN,CADM4_HUMAN,Two-hybrid,21900206,Vinayagam A (2011),CSNK1E_HUMAN
601286,DYNC1I1_HUMAN,CADM4_HUMAN,Two-hybrid,21900206,Vinayagam A (2011),DYNC1I1_HUMAN
1051917,SHMT2_HUMAN,CADM4_HUMAN,Affinity Capture-RNA,22658674,Castello A (2012),SHMT2_HUMAN
1196008,HEPACAM2_HUMAN,CADM4_HUMAN,Affinity Capture-MS,26186194,Huttlin EL (2015),HEPACAM2_HUMAN
2232293,CYP2S1_HUMAN,CADM4_HUMAN,Affinity Capture-MS,28514442,Huttlin EL (2017),CYP2S1_HUMAN


In [109]:
# Extracting the data from the IntAct API using PSCIQUIC

intact_api_url = "http://www.ebi.ac.uk/Tools/webservices/psicquic/intact/webservices"

version = "current"
method = "interactor"
protein = query
format = "tab25"

intact_request_url = f"{intact_api_url}/{version}/search/{method}/{protein}?format={format}"

intact_raw_data = requests.get(intact_request_url).text

print(intact_raw_data)

uniprotkb:A8MVW5-2	uniprotkb:Q8NFZ8	intact:EBI-21700103	intact:EBI-7129521|ensembl:ENSP00000222374.1|uniprotkb:B2R7L5|intact:UNK-5523414|uniprotkb:Q9Y4A4	psi-mi:a8mvw52(display_long)|uniprotkb:HEPACAM2(gene name)|psi-mi:HEPACAM2(display_short)|uniprotkb:MIKI(gene name synonym)|uniprotkb:UNQ305/PRO346(orf name)	psi-mi:cadm4_human(display_long)|uniprotkb:CADM4(gene name)|psi-mi:CADM4(display_short)|uniprotkb:IGSF4C(gene name synonym)|uniprotkb:NECL4(gene name synonym)|uniprotkb:TSLL2(gene name synonym)|uniprotkb:Immunoglobulin superfamily member 4C(gene name synonym)|uniprotkb:Nectin-like protein 4(gene name synonym)|uniprotkb:TSLC1-like protein 2(gene name synonym)	psi-mi:"MI:0007"(anti tag coimmunoprecipitation)	Huttlin et al. (2017)	pubmed:28514442|doi:10.1038/nature22366|imex:IM-25778	taxid:9606(human)|taxid:9606(Homo sapiens)	taxid:9606(human)|taxid:9606(Homo sapiens)	psi-mi:"MI:0914"(association)	psi-mi:"MI:0469"(IntAct)	intact:EBI-21700110|imex:IM-25778-9205	intact-miscore:0.35
un

In [110]:
# Getting the data into a pandas df

# Split each row into a list of columns based on PSI-MI TAB 2.5 format
intact_columns = ['Unique identifier for interactor A', 
                    'Unique identifier for interactor B', 
                    'Alternative identifier for interactor A', 
                    'Alternative identifier for interactor B', 
                    'Aliases for A', 
                    'Aliases for B', 
                    'Interaction detection methods', 
                    'First author', 
                    'Identifier of the publication', 
                    'NCBI Taxonomy identifier for interactor A', 
                    'NCBI Taxonomy identifier for interactor B', 
                    'Interaction types', 
                    'Source databases', 
                    'Interaction identifier(s)', 
                    'Confidence score']

intact_rows = [row.split('\t') for row in intact_raw_data.split('\n')]

# Create a pandas DataFrame from the list of rows and columns
intact_df = pd.DataFrame(intact_rows, columns=intact_columns)
intact_df.shape


(15, 15)

In [112]:
# # Cleaning up the dataframe

# filter out the columns that are needed
intact_df_filter = psicquic_df[['Unique identifier for interactor A', 
                                  'Unique identifier for interactor B', 
                                  'Interaction detection methods', 
                                  'First author', 
                                  'Identifier of the publication', 
                                  'Confidence score']]

# remove the last row
intact_df_filter = intact_df_filter[:-1]

# removing the uniprotkbID refix from the name
intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']] = intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']].map(lambda x: x.removeprefix('uniprotkb:'))

# apply the convert_protein_ID_name function to the first two rows
intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']] = intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']].map(convert_protein_ID_Name)

# removing duplicate rows
df_intact_dupl = intact_df_filter.drop_duplicates()

# rename the column headers, with prefix IntAct
df_intAct_final = df_intact_dupl.rename(columns= {'Unique identifier for interactor A': 'IntAct_interactor_a', 
                                        'Unique identifier for interactor B': 'IntAct_interactor_b',
                                        'Interaction detection methods': 'IntAct_method',
                                        'First author': 'IntAct_publication',
                                        'Identifier of the publication': 'IntAct_pubID',
                                        'Confidence score': 'IntAct_score'})

# get one column, only the interactor of the query
df_IntAct = get_interactors_for_target(df_intAct_final, 'IntAct_interactor_a', 'IntAct_interactor_b', query)

df_IntAct.head(5)

,IntAct_interactor_a,IntAct_interactor_b,IntAct_method,IntAct_publication,IntAct_pubID,IntAct_score,interactor_of_CADM4_HUMAN
0,HECA2_HUMAN,CADM4_HUMAN,"psi-mi:""MI:0007""(anti tag coimmunoprecipitation)",Huttlin et al. (2017),pubmed:28514442|doi:10.1038/nature22366|imex:I...,intact-miscore:0.35,HECA2_HUMAN
1,CD69_HUMAN,CADM4_HUMAN,"psi-mi:""MI:0007""(anti tag coimmunoprecipitation)",Huttlin et al. (2017),pubmed:28514442|doi:10.1038/nature22366|imex:I...,intact-miscore:0.35,CD69_HUMAN
2,CP2S1_HUMAN,CADM4_HUMAN,"psi-mi:""MI:0007""(anti tag coimmunoprecipitation)",Huttlin et al. (2017),pubmed:28514442|doi:10.1038/nature22366|imex:I...,intact-miscore:0.35,CP2S1_HUMAN
3,CD81_HUMAN,CADM4_HUMAN,"psi-mi:""MI:0006""(anti bait coimmunoprecipitation)",Palor et al. (2020),imex:IM-28053|pubmed:32900848,intact-miscore:0.35,CD81_HUMAN
7,CFTR_HUMAN,CADM4_HUMAN,"psi-mi:""MI:1314""(proximity-dependent biotin id...",Chevalier et al. (2022),imex:IM-29540|pubmed:36012204,intact-miscore:0.27,CFTR_HUMAN


In [34]:
# Extracting the data from the STRING-API

string_api_url = "https://string-db.org//api"

output_format = "json"
method = "interaction_partners"

string_request_url = "/".join([string_api_url, output_format, method])

params = {
    "identifiers": query,
    "species": 9606, # human
    "required_score": 0.2,
    "limit": 1000000 
}

response = requests.post(string_request_url, data = params)

string_raw_data = response.text

print(string_raw_data)

[{"stringId_A": "9606.ENSP00000222374", "stringId_B": "9606.ENSP00000222644", "preferredName_A": "CADM4", "preferredName_B": "MPP6", "ncbiTaxonId": 9606, "score": 0.888, "nscore": 0, "fscore": 0, "pscore": 0, "ascore": 0, "escore": 0.045, "dscore": 0, "tscore": 0.888}, {"stringId_A": "9606.ENSP00000222374", "stringId_B": "9606.ENSP00000357106", "preferredName_A": "CADM4", "preferredName_B": "CADM3", "ncbiTaxonId": 9606, "score": 0.862, "nscore": 0, "fscore": 0, "pscore": 0, "ascore": 0.128, "escore": 0, "dscore": 0, "tscore": 0.849}, {"stringId_A": "9606.ENSP00000222374", "stringId_B": "9606.ENSP00000357110", "preferredName_A": "CADM4", "preferredName_B": "EPB41L2", "ncbiTaxonId": 9606, "score": 0.727, "nscore": 0, "fscore": 0, "pscore": 0, "ascore": 0, "escore": 0, "dscore": 0, "tscore": 0.727}, {"stringId_A": "9606.ENSP00000222374", "stringId_B": "9606.ENSP00000345259", "preferredName_A": "CADM4", "preferredName_B": "EPB41", "ncbiTaxonId": 9606, "score": 0.628, "nscore": 0, "fscore":

In [35]:
# Getting the STRING-API data into a pandas df
string_df = pd.read_json(string_raw_data)
string_df.head(5)

/tmp/ipykernel_231037/906031186.py:2: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  string_df = pd.read_json(string_raw_data)


,stringId_A,stringId_B,preferredName_A,preferredName_B,ncbiTaxonId,score,nscore,fscore,pscore,ascore,escore,dscore,tscore
0,9606.ENSP00000222374,9606.ENSP00000222644,CADM4,MPP6,9606,0.888,0,0.0,0.0,0.000,0.045,0,0.888
1,9606.ENSP00000222374,9606.ENSP00000357106,CADM4,CADM3,9606,0.862,0,0.0,0.0,0.128,0.000,0,0.849
2,9606.ENSP00000222374,9606.ENSP00000357110,CADM4,EPB41L2,9606,0.727,0,0.0,0.0,0.000,0.000,0,0.727
3,9606.ENSP00000222374,9606.ENSP00000345259,CADM4,EPB41,9606,0.628,0,0.0,0.0,0.000,0.000,0,0.628
4,9606.ENSP00000222374,9606.ENSP00000264638,CADM4,CNTNAP1,9606,0.610,0,0.0,0.0,0.084,0.000,0,0.592


In [113]:
# Cleaning up the string dataframe

# filter out the columns that are needed
string_df_filter = string_df[[  'preferredName_A', 
                                'preferredName_B', 
                                'score', 
                                'escore']]

# adding 'HUMAN' afterfix to the column values
def add_suffix_human(value):
    return value + '_HUMAN'

string_df_filter[['preferredName_A', 'preferredName_B']] = string_df_filter[['preferredName_A', 'preferredName_B']].apply(add_suffix_human)


# renaming column headers
df_string_final = string_df_filter.rename(columns= {'preferredName_A': 'string_interactor_a', 
                                                    'preferredName_B': 'string_interactor_b',
                                                    'score': 'string_score',
                                                    'escore': 'string_escore'})

# get one column, only the interactor of the query
df_string = get_interactors_for_target(df_string_final, 'string_interactor_a', 'string_interactor_b', query)

df_string.head(5)

/tmp/ipykernel_231037/2941665265.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  string_df_filter[['preferredName_A', 'preferredName_B']] = string_df_filter[['preferredName_A', 'preferredName_B']].apply(add_suffix_human)


,string_interactor_a,string_interactor_b,string_score,string_escore,interactor_of_CADM4_HUMAN
0,CADM4_HUMAN,MPP6_HUMAN,0.888,0.045,MPP6_HUMAN
1,CADM4_HUMAN,CADM3_HUMAN,0.862,0.000,CADM3_HUMAN
2,CADM4_HUMAN,EPB41L2_HUMAN,0.727,0.000,EPB41L2_HUMAN
3,CADM4_HUMAN,EPB41_HUMAN,0.628,0.000,EPB41_HUMAN
4,CADM4_HUMAN,CNTNAP1_HUMAN,0.610,0.000,CNTNAP1_HUMAN


In [68]:
# GETTING THE DATA FROM the HIPPIE-API

hippie_api_url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/queryHIPPIE.php"

protein_to_query = "CADM4"
layer = 1 #to query protein within input set (0) or against all HIPPIE proteins (1, default)
threshold = 0 #confidence threshold, default is 0
format = "conc_file" #this generates a tab seperated text file (other interesting input types: "mitab", "browser")

hippie_request_url = f"{hippie_api_url}?proteins={protein_to_query}&layers={layer}&conf_thres={threshold}&out_type={format}"

hippie_response = requests.get(hippie_request_url).text

print(hippie_response)

print(hippie_request_url)

# there seems to be an issue with the PHP request response, probably the server is not correctly configured

<?php

$url = "http" . (isset($_SERVER["HTTPS"]) ? "s" : "") . "://" . "{$_SERVER['HTTP_HOST']}/{$_SERVER['REQUEST_URI']}";

$parts = parse_url($url);
parse_str($parts['query'], $query);

$out_type = empty($query['out_type']) ? "conc_file" : $query['out_type'];
$query_genes = empty($query['proteins']) ? "" : str_replace(array(",", ";", "|"), PHP_EOL, $query['proteins']);
$layers = empty($query['layers']) ? 1 : $query['layers'];
$layers = $query['layers']=='0' ? 0 : $layers;
$conf_thres = empty($query['conf_thres']) ? 0 : $query['conf_thres'];

echo '<html>
         <form action="fast_query_tissue.php" method="POST" id="netQuery">
            <input type="hidden" name="out_type" value="'.$out_type.'" />
            <input type="hidden" name="query_genes" value="'.$query_genes.'" />
            <input type="hidden" name="layers" value='.$layers.' />
            <input type="hidden" name="conf_thres" value='.$conf_thres.' />
         </form>
         <script type="text/javascript">
      

In [72]:
## !!!!! HIPPIE website seems to be down at the moment

# GETTING THE DATA FROM HIPPIE through WEBSCRAPING
import requests
import json
import re
from bs4 import BeautifulSoup

protein = query

url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/query.php?s="+str(protein)

payload = {}
headers = {
'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7', 
'Accept-Language': 'nl-NL,nl;q=0.9,en-US;q=0.8,en;q=0.7,fr;q=0.6' ,
'Connection': 'keep-alive', 
'Referer': 'http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/',
'Upgrade-Insecure-Requests': '1' ,
'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Mobile Safari/537.36' 
}

response = requests.request("GET", url, headers=headers, data=payload)
soup = BeautifulSoup(response.text, "html.parser")
table = soup.find('tbody') # already skipped the columns names

rows = table.find_all('tr')

interactions = []
for idx, row in enumerate(rows):
    data = row.find_all("td")
    interaction = {
    "Interactor": data[0].text,
    "EntrezGeneID": data[1].text,
    "GeneSymbol": data[2].text,
    "Score": data[3].text
    }
    interactions.append(interaction) 
   

print(rows)


[<tr><td>".$int-&gt;getLinkFormattedUniprotIDOfInteractor2()."</td><td>".$int-&gt;getLinkFormattedEntrezIDOfInteractor2()."</td><td>"
                     .$int-&gt;getNamesOfInteractor2()."</td><td><a "&i1="$int-" echo="" entrez'){="" href='details.php?q=$int-&gt;dbId";
                     if($type == '>entrezGeneId1&amp;i2=".$int-&gt;entrezGeneId2[0]."&amp;t=e";
                     }
                     elseif($type == 'uniprot'){
                        echo "&amp;i1=$int-&gt;uniprotId1&amp;i2=$int-&gt;uniprotId2&amp;t=u";
                     }
                     echo "'&gt;".$int-&gt;score."</a></td>";
                     echo "<td><a href='query.php?s=".$int-&gt;uniprotId2."'>Show</a></td></tr>]


In [77]:
# GETTING THE DATA FROM THE APID db by WEBSCRAPING

import requests
import json
import re
from bs4 import BeautifulSoup


#Function gets protein id from name value. (Example: VCAM1_HUMAN -> P19320). We will need this protein ID to generate our table.
def extract_protein_id(name):
    url = "http://cicblade.dep.usal.es:8080/APID/searchProtein.action" # Endpoint to search for protein by name

    payload = 'proteinName='+str(name)+'&taxon=0'
    headers = {
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Language': 'en-US,en;q=0.9',
    'Cache-Control': 'max-age=0',
    'Connection': 'keep-alive',
    'Content-Type': 'application/x-www-form-urlencoded',
    'Cookie': 'JSESSIONID=086030D12C94A8248DA2B5B9A84C16FA; _ga=GA1.2.581619552.1696441740; _gid=GA1.2.818200239.1696441740; _gat=1; _ga_7JSDHY18SK=GS1.2.1696441740.1.1.1696443448.0.0.0',
    'Origin': 'http://cicblade.dep.usal.es:8080',
    'Referer': 'http://cicblade.dep.usal.es:8080/APID/init.action',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
    }

    response = requests.request("POST", url, headers=headers, data=payload) # Execute HTTP request, response is stored in 'response'
    soup = BeautifulSoup(response.text, "html.parser") #Parse response.text (which is HTML) with a HTML parser
    table = soup.find('table', id="proteins") #use soup.find function to find a table in the HTML with id "proteins" (You can get this value by checking the HTML in the response from your HTTP request above. Hardcoded it to "proteins" because this will not change.)
    row = table.find_next('td') #use the beautifulsoup library again to find a html element in the found table. (Give me the first column value.)
    
    return row.text # row.text gives us the protein ID, we will need this protein ID to generate our table. (Example: VCAM1_HUMAN -> P19320)


def extract_table(proteinid):
    url = "http://cicblade.dep.usal.es:8080/APID/InteractionsGrid.action?protein1="+str(proteinid)+"&protein2=NA"

    payload = {}
    headers = {
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Language': 'en-US,en;q=0.9',
    'Connection': 'keep-alive',
    'Cookie': 'JSESSIONID=086030D12C94A8248DA2B5B9A84C16FA; _ga=GA1.2.581619552.1696441740; _gid=GA1.2.818200239.1696441740; _ga_7JSDHY18SK=GS1.2.1696441740.1.1.1696442421.0.0.0',
    'Referer': 'http://cicblade.dep.usal.es:8080/APID/searchProtein.action',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
    }

    response = requests.request("GET", url, headers=headers, data=payload)
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find('table', id="interactions") #Find the table

    rows = table.find_all("tr") # Find all table rows
    interactions = [] #initialize empty list
    for idx, row in enumerate(rows): #loop over rows, keep index
        if idx == 0: # first row is the header, skip.
            pass
        else:
            try:
                data1 = row.find_all("td") # get column
                interaction = { # build intraction object
                "ProteinA": data1[0].get_text().strip(),
                "ProteinB": data1[1].get_text().strip(),
                "MethodType": data1[2].get_text().strip(),
                "Method": data1[3].get_text().strip(),
                "Publication": re.sub(' +', ' ',data1[4].get_text().strip().replace("\n", "")),
                "Source": data1[5].get_text().strip()
                }
                interactions.append(interaction) #append interaction object to result list
            except:
                pass
    return interactions #return the result list


proteinid = extract_protein_id(query)
apid_results = extract_table(proteinid)
# print(json.dumps(apid_results, indent=4))
print(apid_results)

[{'ProteinA': 'CADM4_HUMAN', 'ProteinB': 'KC1E_HUMAN', 'MethodType': 'binary', 'Method': 'two hybrid (MI:0018)', 'Publication': 'Vinayagam, A. et al., 2011 (PMID:21900206 )', 'Source': 'IntAct (Acc: EBI-7129549  )'}, {'ProteinA': 'CADM4_HUMAN', 'ProteinB': 'KC1E_HUMAN', 'MethodType': 'binary', 'Method': 'two hybrid (MI:0018)', 'Publication': 'Vinayagam, A. et al., 2011 (PMID:21900206 )', 'Source': 'BioGRID (Acc: 600990  )'}, {'ProteinA': 'CADM4_HUMAN', 'ProteinB': 'DC1I1_HUMAN', 'MethodType': 'binary', 'Method': 'two hybrid (MI:0018)', 'Publication': 'Vinayagam, A. et al., 2011 (PMID:21900206 )', 'Source': 'IntAct (Acc: EBI-7145902  )'}, {'ProteinA': 'CADM4_HUMAN', 'ProteinB': 'DC1I1_HUMAN', 'MethodType': 'binary', 'Method': 'two hybrid (MI:0018)', 'Publication': 'Vinayagam, A. et al., 2011 (PMID:21900206 )', 'Source': 'BioGRID (Acc: 601286  )'}, {'ProteinA': 'CADM4_HUMAN', 'ProteinB': 'ACTS_HUMAN', 'MethodType': 'indirect', 'Method': 'affinity chromatography technology (MI:0004)', 'Pu

In [114]:
# Getting the data from APID into a df

apid_df = pd.DataFrame(apid_results)

# filter out the columns that are needed
apid_df_filter = apid_df[[  'ProteinA', 
                                'ProteinB', 
                                'Method', 
                                'Publication',
                                'Source']]

# renaming column headers
df_apid_final = apid_df_filter.rename(columns= {'ProteinA': 'apid_interactor_a', 
                                                    'ProteinB': 'apid_interactor_b',
                                                    'Method': 'apid_method',
                                                    'Publication': 'apid_publication',
                                                    'Source': 'apid_source'})

# get one column, only the interactor of the query
df_apid = get_interactors_for_target(df_apid_final, 'apid_interactor_a', 'apid_interactor_b', query)

df_apid.head(5)

,apid_interactor_a,apid_interactor_b,apid_method,apid_publication,apid_source,interactor_of_CADM4_HUMAN
0,CADM4_HUMAN,KC1E_HUMAN,two hybrid (MI:0018),"Vinayagam, A. et al., 2011 (PMID:21900206 )",IntAct (Acc: EBI-7129549 ),KC1E_HUMAN
1,CADM4_HUMAN,KC1E_HUMAN,two hybrid (MI:0018),"Vinayagam, A. et al., 2011 (PMID:21900206 )",BioGRID (Acc: 600990 ),KC1E_HUMAN
2,CADM4_HUMAN,DC1I1_HUMAN,two hybrid (MI:0018),"Vinayagam, A. et al., 2011 (PMID:21900206 )",IntAct (Acc: EBI-7145902 ),DC1I1_HUMAN
3,CADM4_HUMAN,DC1I1_HUMAN,two hybrid (MI:0018),"Vinayagam, A. et al., 2011 (PMID:21900206 )",BioGRID (Acc: 601286 ),DC1I1_HUMAN
4,CADM4_HUMAN,ACTS_HUMAN,affinity chromatography technology (MI:0004),"Huttlin, EL. et al., 2017 (PMID:28514442 )",BioPlex,ACTS_HUMAN


In [ ]:
# Make an intersecions diagram using the pyUpSet to check the interactors of VCAM1 
from upsetplot import UpSet


